In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Custom Symmetric Mask Layer
class SymmetricMaskLayer(Layer):
    def __init__(self, input_dim, **kwargs):
        super(SymmetricMaskLayer, self).__init__(**kwargs)
        self.input_dim = input_dim

    def build(self, input_shape):
        self.mask_logits = self.add_weight(shape=(self.input_dim,),
                                           initializer='zeros',
                                           trainable=True,
                                           name='mask_logits')
        super(SymmetricMaskLayer, self).build(input_shape)

    def call(self, inputs):
        mask = tf.sigmoid(self.mask_logits)
        return inputs * mask

    def get_mask(self):
        return tf.sigmoid(self.mask_logits)

# Number of input features
input_dim = 45

# Input layer
inputs = Input(shape=(input_dim,))

# Instantiate the SymmetricMaskLayer
mask_layer = SymmetricMaskLayer(input_dim)

# Apply the mask to the inputs
masked_inputs = mask_layer(inputs)

# Encoder
encoded = Dense(32, activation='relu')(masked_inputs)
encoded = Dense(16, activation='relu')(encoded)
encoded = Dense(8, activation='relu')(encoded)  # Bottleneck

# Decoder
decoded = Dense(16, activation='relu')(encoded)
decoded = Dense(32, activation='relu')(decoded)
# Output before applying the mask
decoded_outputs = Dense(input_dim, activation='sigmoid')(decoded)

# Apply the same mask to the outputs
masked_outputs = mask_layer(decoded_outputs)

# Autoencoder Model
autoencoder = Model(inputs=inputs, outputs=masked_outputs)

# Reconstruction loss
def reconstruction_loss(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

# Regularization loss
def mask_regularization(mask_values):
    l1_reg = tf.reduce_sum(tf.abs(mask_values))
    binary_reg = tf.reduce_sum(mask_values * (1 - mask_values))
    return l1_reg + binary_reg

# Total loss
def total_loss(y_true, y_pred):
    mask_values = mask_layer.get_mask()
    recon_loss = reconstruction_loss(y_true, y_pred)
    reg_loss = mask_regularization(mask_values)
    λ = 0.01
    return recon_loss + λ * reg_loss

# Compile the model
autoencoder.compile(optimizer=Adam(), loss=total_loss)

# Train the model
autoencoder.fit(X_train, X_train, epochs=50, batch_size=32, validation_split=0.2)

# Extract the mask
learned_mask_values = mask_layer.get_mask().numpy()
threshold = 0.5
binary_mask = (learned_mask_values > threshold).astype(int)

print("Learned Mask Values:")
print(learned_mask_values)

print("Binary Mask (Selected Channels):")
print(binary_mask)


By applying the same mask to both the input and output layers, you ensure that the selection of channels is consistent throughout the autoencoder. This approach allows you to:

1) Select Specific Channels at the Input: The mask controls which channels (sensors) are considered by the encoder.
2) Maintain Correspondence in the Output: The same mask ensures that the decoder's output aligns with the masked input, selecting the same channels.
3) Learn the Mask During Training: The mask is a trainable parameter, allowing the model to learn which channels are most important for reconstruction.
4) Interpret the Mask: After training, you can interpret the mask to understand which sensors are critical.


However, a binary mask (values exactly 0 or 1) during training can cause problems with gradient flow during backpropagation. Specifically, when the mask value is 0, it blocks the gradient from flowing through that path, preventing the network from learning whether that input channel could be useful if included.

To avoid this problem, we can use techniques that allow gradients to flow even when the mask appears binary during the forward pass. Two common methods are:

1) Straight-Through Estimator (STE): Use a binary mask in the forward pass but approximate gradients during backpropagation to allow learning.
2) Gumbel-Softmax Trick: Create a differentiable approximation to sampling from a discrete distribution.
Below, I'll show you how to modify your code using the Straight-Through Estimator (STE) to solve this problem.



**Modified Code Using Straight-Through Estimator**

1. Update the SymmetricMaskLayer
We'll modify the SymmetricMaskLayer to implement the STE. In this layer:

1) Forward Pass: We sample a binary mask based on the probabilities derived from mask_logits.
2) Backward Pass: We allow gradients to flow through the continuous probabilities, even though we're using a binary mask in the forward pass.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Layer, Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Custom Symmetric Mask Layer with STE
class SymmetricMaskLayer(Layer):
    def __init__(self, input_dim, **kwargs):
        super(SymmetricMaskLayer, self).__init__(**kwargs)
        self.input_dim = input_dim

    def build(self, input_shape):
        # Initialize mask logits
        self.mask_logits = self.add_weight(shape=(self.input_dim,),
                                           initializer='zeros',
                                           trainable=True,
                                           name='mask_logits')
        super(SymmetricMaskLayer, self).build(input_shape)

    def call(self, inputs):
        # Compute probabilities using sigmoid
        probs = tf.sigmoid(self.mask_logits)
        # Sample binary mask using Bernoulli sampling
        random_uniform = tf.random.uniform(tf.shape(probs))
        binary_mask = tf.cast(tf.less(random_uniform, probs), tf.float32)
        # Use STE: during backprop, replace binary_mask with probs
        mask = binary_mask + probs - tf.stop_gradient(probs)
        return inputs * mask

    def get_mask(self):
        # Return the probabilities
        return tf.sigmoid(self.mask_logits)


Explanation:

probs: Continuous probabilities between 0 and 1.

binary_mask: Sampled binary mask (0 or 1) used during the forward pass.

mask: Combines binary_mask with probs to allow gradients to flow during backpropagation.

**2. Build the Autoencoder Model**

We use the modified SymmetricMaskLayer in the autoencoder architecture.

In [ ]:
# Number of input features
input_dim = 45

# Input layer
inputs = Input(shape=(input_dim,))

# Instantiate the SymmetricMaskLayer
mask_layer = SymmetricMaskLayer(input_dim)

# Apply the mask to the inputs
masked_inputs = mask_layer(inputs)

# Encoder
encoded = Dense(32, activation='relu')(masked_inputs)
encoded = Dense(16, activation='relu')(encoded)
encoded = Dense(8, activation='relu')(encoded)  # Bottleneck

# Decoder
decoded = Dense(16, activation='relu')(encoded)
decoded = Dense(32, activation='relu')(decoded)
# Output before applying the mask
decoded_outputs = Dense(input_dim, activation='sigmoid')(decoded)

# Apply the same mask to the outputs
masked_outputs = mask_layer(decoded_outputs)

# Autoencoder Model
autoencoder = Model(inputs=inputs, outputs=masked_outputs)


In [ ]:

def reconstruction_loss(y_true, y_pred):
    return tf.reduce_mean(tf.square(y_true - y_pred))

# Regularization loss to encourage sparsity and binarity
def mask_regularization(probs):
    # L1 regularization to promote sparsity
    l1_reg = tf.reduce_sum(probs)
    # Binarization regularization
    binary_reg = tf.reduce_sum(probs * (1 - probs))
    return l1_reg + binary_reg

# Total loss
def total_loss(y_true, y_pred):
    probs = mask_layer.get_mask()
    recon_loss = reconstruction_loss(y_true, y_pred)
    reg_loss = mask_regularization(probs)
    λ = 0.01  # Regularization coefficient
    return recon_loss + λ * reg_loss
 in the autoencoder architecture.

Explanation:

reconstruction_loss: Measures how well the autoencoder reconstructs the input.

mask_regularization: Encourages the mask probabilities to be sparse and close to binary values (0 or 1).

λ: Controls the trade-off between reconstruction accuracy and mask regularization.

**Alternative Solution Using Gumbel-Softmax**

If you prefer, you can use the Gumbel-Softmax trick to approximate the sampling of discrete masks while maintaining differentiability.

Modified SymmetricMaskLayer with Gumbel-Softmax

In [ ]:
# Custom Symmetric Mask Layer with Gumbel-Softmax
class SymmetricMaskLayer(Layer):
    def __init__(self, input_dim, temperature=0.5, **kwargs):
        super(SymmetricMaskLayer, self).__init__(**kwargs)
        self.input_dim = input_dim
        self.temperature = temperature

    def build(self, input_shape):
        self.mask_logits = self.add_weight(shape=(self.input_dim,),
                                           initializer='zeros',
                                           trainable=True,
                                           name='mask_logits')
        super(SymmetricMaskLayer, self).build(input_shape)

    def call(self, inputs):
        # Sample from Gumbel distribution
        uniform_noise = tf.random.uniform(tf.shape(self.mask_logits), minval=0, maxval=1)
        gumbel_noise = -tf.math.log(-tf.math.log(uniform_noise + 1e-20) + 1e-20)
        # Compute the Gumbel-Softmax sample
        logits = self.mask_logits + gumbel_noise
        mask = tf.sigmoid(logits / self.temperature)
        return inputs * mask

    def get_mask(self):
        # Return the probabilities
        return tf.sigmoid(self.mask_logits)
